In [ ]:
# =============================================================================
# CELL 1: CONFIGURATION, BASELINES, POOL DEFINITIONS
# =============================================================================

granularity = 'm'

START_DATE = '2025-10-07'
END_DATE = None

run_every_query = True

DATE_COL_MAP = {'q': 'book_date', 'm': 'book_date', 'w': 'app_date'}

LOBS = ['STE']

BASELINES = {
    'STE': {'ltv': 1.94, 'new_recovery_unadjusted': 0.557, 'apr': 0.25},
}

MODEL_PARAMS = {
    'mmi_standard_increase': 1.03,
    'expected_years_on_book': 2,
    'impound_probability': 0.15,
    'mean_unit_loss': 0.5,
    'unit_loss_to_model_score': 0.02,
    'skip_rate': 0.75,
    'kmx_loss_scale': 0.067,
}

PRICING_SCALAR = 1.0

USE_STE_METRICS = True

EXCLUDED_VINTAGES = {}

MODEL_SCORE_THRESHOLD = 140

POOLS = {
    'Vanilla':         {'ula_filter': None, 'weekly_filter': None},
    'New Cars':        {'ula_filter': lambda df: df[df['is_new'] == 1],
                        'weekly_filter': lambda df: df[df['is_new'] == 1]},
    'Downcredit Used': {'ula_filter': lambda df: df[(df['is_new'] == 0) & (df['model_score'] < MODEL_SCORE_THRESHOLD)],
                        'weekly_filter': lambda df: df[(df['is_new'] == 0) & (df['model_score'] < MODEL_SCORE_THRESHOLD)]},
    'Upcredit Used':   {'ula_filter': lambda df: df[(df['is_new'] == 0) & (df['model_score'] >= MODEL_SCORE_THRESHOLD)],
                        'weekly_filter': lambda df: df[(df['is_new'] == 0) & (df['model_score'] >= MODEL_SCORE_THRESHOLD)]},
    'Downcredit':      {'ula_filter': lambda df: df[df['model_score'] < MODEL_SCORE_THRESHOLD],
                        'weekly_filter': lambda df: df[df['model_score'] < MODEL_SCORE_THRESHOLD]},
    'Upcredit':        {'ula_filter': lambda df: df[df['model_score'] >= MODEL_SCORE_THRESHOLD],
                        'weekly_filte
                        
                        
                        
                        r': lambda df: df[df['model_score'] >= MODEL_SCORE_THRESHOLD]},
}

print(f"Pools: {list(POOLS.keys())}")
print(f"Model score threshold: {MODEL_SCORE_THRESHOLD}")

Pools: ['Vanilla', 'New Cars', 'Downcredit Used', 'Upcredit Used', 'Downcredit', 'Upcredit']
Model score threshold: 140


In [2]:
# =============================================================================
# CELL 2: IMPORTS AND DERIVED CONFIGURATION
# =============================================================================
import pandas as pd
import numpy as np
import pyodbc
import pickle
import warnings
import time
import datetime as dt
import re
import os
import openpyxl
from tqdm.notebook import tqdm

tqdm.pandas()
pd.set_option('display.max_columns', 100)
pd.set_option('display.min_rows', 100)

date_col = DATE_COL_MAP[granularity]
start_date = pd.Timestamp(START_DATE)
end_date = pd.Timestamp(END_DATE) if END_DATE else pd.Timestamp.today().normalize()

PERIOD_FREQ_MAP = {'q': 'Q', 'm': 'M', 'w': 'W-SAT'}
period_freq = PERIOD_FREQ_MAP[granularity]

start_period = start_date.to_period(period_freq)
end_period = end_date.to_period(period_freq)

min_date_sql = f"'{START_DATE}'"

print(f"Granularity: {granularity}")
print(f"Date column: {date_col}")
print(f"Period range: {start_period} to {end_period}")
print(f"SQL min_date: {min_date_sql}")

Granularity: m
Date column: book_date
Period range: 2025-10 to 2026-08
SQL min_date: '2025-10-07'


In [3]:
# =============================================================================
# CELL 3: UTILITY FUNCTIONS
# =============================================================================

def run_sql(filename, sub_list=None, connection=None, filename_is_query=False):
    if sub_list is None:
        sub_list = []
    if filename_is_query:
        query = filename
    else:
        with open(filename, 'r') as file:
            query = file.read()
    for text, var in sub_list:
        query = query.replace(text, var)
    if connection is None:
        with pyodbc.connect("DSN=Redshift_prod_new") as conn:
            warnings.filterwarnings("ignore", category=UserWarning)
            df = pd.read_sql_query(sql=query, con=conn)
            warnings.filterwarnings("default", category=UserWarning)
            return df
    else:
        warnings.filterwarnings("ignore", category=UserWarning)
        df = pd.read_sql_query(sql=query, con=connection)
        warnings.filterwarnings("default", category=UserWarning)
        return df


def store_pickle(data, filename):
    if isinstance(data, str):
        data, filename = filename, data
    with open(filename, 'wb') as file:
        pickle.dump(data, file)


def get_pickle(filename):
    with open(filename, 'rb') as file:
        return pickle.load(file)


def cached_sql(query, pickle_name, sub_list=None, connection=None,
               force_refresh=False, filename_is_query=False):
    if force_refresh or not os.path.exists(pickle_name):
        df = run_sql(query, sub_list=sub_list, connection=connection,
                     filename_is_query=filename_is_query)
        if df.empty:
            print(f"  WARNING: query returned 0 rows")
        store_pickle(df, pickle_name)
        return df
    df = get_pickle(pickle_name)
    if df.empty:
        print(f"  WARNING: cached '{pickle_name}' contains 0 rows")
    return df


def smooth(series):
    averaged_series = pd.Series(index=series.index, dtype=float)
    averaged_series.iloc[0] = series.iloc[0]
    averaged_series.iloc[1] = (series.iloc[0] + series.iloc[1] + series.iloc[2]) / 3
    for i in range(2, len(series) - 2):
        averaged_series.iloc[i] = series.iloc[i-2:i+3].mean()
    averaged_series.iloc[-2] = (series.iloc[-1] + series.iloc[-2] + series.iloc[-3]) / 3
    averaged_series.iloc[-1] = series.iloc[-1]
    return averaged_series


def weight_by_proceeds(metric, proceeds):
    return (metric * proceeds).sum() / proceeds.sum()


def weighted_average_and_sum(group, metrics):
    if isinstance(metrics, str):
        weighted_avg = (group[metrics] * group.amt_financed).sum() / group.amt_financed.sum()
        return pd.Series({metrics: weighted_avg, 'amt_financed': group.amt_financed.sum()})
    result_dict = {'amt_financed': group.amt_financed.sum()}
    for metric in metrics:
        weighted_avg = (group[metric] * group.amt_financed).sum() / group.amt_financed.sum()
        result_dict[metric] = weighted_avg
    return pd.Series(result_dict)


def assign_period(df, col_name, freq):
    dt_series = pd.to_datetime(df[col_name])
    return dt_series.dt.to_period(freq)


def format_vintage(period_series):
    if period_series.empty:
        return period_series.astype(str)
    freq = period_series.iloc[0].freqstr
    if freq == 'Q-DEC':
        return period_series.dt.year.astype(str) + ' Q' + period_series.dt.quarter.astype(str)
    elif freq == 'M':
        return period_series.dt.year.astype(str) + ' M' + period_series.dt.month.astype(str).str.zfill(2)
    return period_series.astype(str)


def _ste_weighted_avg(g, metric_col, weight_col='con_amount_financed_back'):
    mask = g[metric_col].notna()
    if not mask.any():
        return np.nan
    return (g.loc[mask, metric_col] * g.loc[mask, weight_col]).sum() / g.loc[mask, weight_col].sum()

print("Utilities ready")

Utilities ready


In [4]:
# =============================================================================
# CELL 4: ULA MULTIPLIER FUNCTIONS
# =============================================================================

def get_ula_multiplier_nonkmx(ula_df, leave_out='None'):
    ula_df['loss_multiplier'] = 1.0

    if leave_out != 'Previous ACA chargeoff':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.prev_co_flag

    if leave_out != 'Small amount financed':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.small_amt_financed_flag * (1 - ula_df.pricing_change_flag)

    if leave_out != 'Zero cash down':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.zero_cash_down_flag * (1 - ula_df.pricing_change_flag)

    if leave_out != 'High mileage vehicle':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.high_mileage_vehicle_flag

    if leave_out != 'High PTI':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.high_pti_flag * (1 - ula_df.pricing_change_flag)

    if leave_out != 'Car make':
        ula_df.loss_multiplier *= (1 + 0.1 * ula_df.car_make_penalty_flag
                                   - 0.1 * ula_df.car_make_benefit_flag
                                   - 0.1 * ula_df.pricing_change_flag * ula_df.car_make_benefit_flag)

    if leave_out != 'Theft risk':
        ula_df.loss_multiplier *= (0.987 + 0.099 * ula_df.theft_risk_flag * (1 - ula_df.pricing_change_flag)
                                   + 0.013 * ula_df.pricing_change_flag)

    if leave_out != 'MCY high model score, low mileage':
        ula_df.loss_multiplier *= 1 - 0.2 * ula_df.mcy_low_mileage_flag

    if leave_out != 'Weekday/weekend decision':
        ula_df.loss_multiplier *= 1 - 0.05 * ula_df.weekend_flag + 0.02 * ula_df.weekday_flag

    if leave_out != 'Student Loans':
        ula_df.loss_multiplier *= 1 + (0.1 * ula_df.student_loan_flag
            - np.minimum(np.maximum((ula_df.cd_model_score - (130 - 2 * ula_df.ent_flag)) * 0.006, 0), 0.03)
            * (1 - ula_df.student_loan_flag)) * ula_df.student_loans_cutoff_date

    if leave_out != 'Low PTI':
        ula_df.loss_multiplier *= 1 + (-0.03 * np.minimum(ula_df.cd_model_score, 135) + 3.9) * ula_df.low_pti_flag

    if leave_out != 'Secured credit (Chime, etc.)':
        ula_df.loss_multiplier *= 0.966 + 0.273 * ula_df.nonkmx_chime_flag

    if leave_out != 'Employment type':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.seasonal_employment_flag - 0.1 * ula_df.waiter_employment_flag

    if leave_out != 'Authorized tradelines':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.nonkmx_auth_tradelines_flag

    if leave_out != 'Fraud':
        ula_df.loss_multiplier *= 1 + (ula_df.fraud_adjustment - 1)

    if leave_out != 'Driver flag':
        ula_df.loss_multiplier *= 1 + 0.15 * ula_df.driver_flag

    if leave_out != 'Clip':
        ula_df.loss_multiplier = np.clip(ula_df.loss_multiplier, 0.8, 2)

    if leave_out != 'Dealer Level (Non-KMX)':
        ula_df.loss_multiplier *= ula_df.pricing_scalar
        ula_df.loss_multiplier = np.clip(ula_df.loss_multiplier, 0.7, 1.4)

    return ula_df


def get_ula_multiplier_kmx(ula_df, loss_scale=None, leave_out='None'):
    if loss_scale is None:
        loss_scale = MODEL_PARAMS['kmx_loss_scale']

    ula_df['loss_multiplier'] = 1.0

    if leave_out != 'Low FICO':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + 0.25 * (ula_df.low_fico_flag & ~ula_df.high_model_score_flag)
            + loss_scale * (ula_df.low_fico_flag & ula_df.high_model_score_flag))

    if leave_out != 'Low Vantage':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + 0.25 * (ula_df.low_vantage_flag & ~ula_df.high_model_score_flag)
            + loss_scale * (ula_df.low_vantage_flag & ula_df.high_model_score_flag))

    if leave_out != 'High PTI':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + 0.05 * ula_df.high_pti_tier_1_flag
            + 0.1 * ula_df.high_pti_tier_2_flag
            + (1.4 * (1 + loss_scale) - 1) * ula_df.high_pti_tier_3_flag)

    ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] /= (1 + loss_scale)

    if leave_out != 'Secured credit (Chime, etc.)':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + (-0.01 + 0.06 * ula_df.secured_credit_flag)
            * ~(~ula_df.job_time_flag & ula_df.narrowed_soft_pull_flag & ula_df.secured_credit_flag))

    if leave_out != 'Authorized tradelines':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= 0.99 + 0.18 * ula_df.kmx_auth_tradelines_flag

    if leave_out != 'Soft pull':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + ~ula_df.job_time_flag * (~ula_df.narrowed_soft_pull_flag * -0.025
            + ula_df.narrowed_soft_pull_flag * (~ula_df.secured_credit_flag * 0.108
            + ula_df.secured_credit_flag * 0.295)))

    if leave_out != 'Fraud':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + (ula_df.fraud_adjustment - 1)

    if leave_out != 'Clip':
        mask = (ula_df.mtn_model.isin([3.0])) & (~ula_df.high_pti_tier_3_flag)
        ula_df.loc[mask, 'loss_multiplier'] = np.clip(ula_df.loc[mask, 'loss_multiplier'], 0.8, 1.35 / (1 + loss_scale))

    if leave_out != 'Vehicle Age':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= 0.8593 + 0.0201 * ula_df.continuous_vehicle_age

    if leave_out != 'npc':
        ula_df.loc[ula_df.kmx_npc_flag, 'loss_multiplier'] *= 0.99 + 0.11 * ula_df.high_pti_npc

    if leave_out != 'Student Loans':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.96 + 0.14 * ula_df.student_loan_flag

    if leave_out != 'high sales price':
        ula_df.loc[ula_df.mtn_model.isin([4.1]), 'loss_multiplier'] *= 0.98 + 0.22 * ula_df.high_sales_price_flag

    if leave_out != 'Driver flag':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.15 * ula_df.driver_flag

    if leave_out != 'Louisiana':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.35 * ula_df.louisiana_flag

    if leave_out != 'georgia':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.1 * ula_df.georgia_flag

    if leave_out != 'txca':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]) & (ula_df.cd_model_score > 140), 'loss_multiplier'] *= 1 - 0.1 * ula_df.txca_flag
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]) & (ula_df.cd_model_score <= 140) & (ula_df.cd_model_score >= 135), 'loss_multiplier'] *= 1 - 0.05 * ula_df.txca_flag

    if leave_out != 'state_counter_adj':
        mask = ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]) & (ula_df.cd_model_score < 150) & ~(ula_df.louisiana_flag | ula_df.georgia_flag | ula_df.txca_flag)
        ula_df.loc[mask, 'loss_multiplier'] *= 1.012

    if leave_out != 'Secured credit (Chime, etc.)':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= (
            0.96 + (0.01 * ula_df.soft_pull_flag - 0.18 * ula_df.chime_flag * ula_df.soft_pull_flag) + 0.46 * ula_df.chime_flag)

    if leave_out != 'Job time':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.99 + 0.21 * ula_df.job_time_flag

    if leave_out != 'Existing DQ':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.99 + 0.11 * ula_df.existing_dq_flag

    if leave_out != 'Employment type':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.1 * ula_df.seasonal_employment_flag

    if leave_out != 'Authorized tradelines':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.99 + 0.06 * ula_df.kmx_auth_tradelines_flag

    if leave_out != 'Soft pull':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]) & ula_df.soft_pull_flag, 'loss_multiplier'] *= 1.1 * (0.99 + 0.11 * ula_df.low_bureau_flag) * (0.978 + 0.172 * ula_df.open_tl_flag)
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]) & ~ula_df.soft_pull_flag, 'loss_multiplier'] *= (1 * (0.98 + 0.22 * ula_df.low_bureau_flag) * (0.945 + 0.405 * ula_df.open_tl_flag) / np.maximum(ula_df.cd_perc_flag * ula_df.open_tl_flag * 1.2, 1))

        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2]) & ula_df.soft_pull_flag, 'loss_multiplier'] *= 0.97 + 0.15 * ula_df.cd_perc_flag
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2]) & ~ula_df.soft_pull_flag, 'loss_multiplier'] *= 0.954 + 0.346 * ula_df.cd_perc_flag
        ula_df.loc[(ula_df.mtn_model == 4.1) & ula_df.soft_pull_flag, 'loss_multiplier'] *= 1.0 + 0.05 * ula_df.cd_perc_flag
        ula_df.loc[(ula_df.mtn_model == 4.1) & ~ula_df.soft_pull_flag, 'loss_multiplier'] *= 1.0 + 0.10 * ula_df.cd_perc_flag

    if leave_out != 'blanket adjustment':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'] /= 1.1

    if leave_out != 'Clip':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'] = np.clip(
            ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'], 0.75, 1.4)

    return ula_df

print("ULA multiplier functions ready")

ULA multiplier functions ready


In [5]:
# =============================================================================
# CELL 5: DATA FETCH (SQL FROM TXT FILES)
# =============================================================================

os.makedirs('cache', exist_ok=True)
force = run_every_query

CACHE_ULA = '../../cache/ste_vanillas_ula.pkl'
CACHE_RECOVERY = '../../cache/ste_vanillas_recovery.pkl'
CACHE_WEEKLY = '../../cache/ste_vanillas_weekly.pkl'

need_conn = force or not all(
    os.path.exists(p) for p in (CACHE_ULA, CACHE_RECOVERY, CACHE_WEEKLY)
)

if need_conn:
    conn = pyodbc.connect("DSN=Redshift_prod_new")

    with open('../queries/ste_ragu_temptables.txt', 'r') as f:
        conn.execute(f.read().strip())
    print('Temp tables created')

    ula_df_total = cached_sql('../queries/ste_ragu_ula.txt', CACHE_ULA,
                              connection=conn, force_refresh=force)
    print(f'ULA ready: {len(ula_df_total):,} records')

    new_recovery = cached_sql('../queries/ste_ragu_recovery.txt', CACHE_RECOVERY,
                              connection=conn, force_refresh=force)
    print(f'Recovery ready: {len(new_recovery):,} records')

    ste_weekly_raw = cached_sql('../queries/ste_ragu_weekly.txt', CACHE_WEEKLY,
                                connection=conn, force_refresh=force)
    print(f'STE weekly metrics ready: {len(ste_weekly_raw):,} records')

    conn.close()
else:
    ula_df_total = get_pickle(CACHE_ULA)
    new_recovery = get_pickle(CACHE_RECOVERY)
    ste_weekly_raw = get_pickle(CACHE_WEEKLY)
    print('All data loaded from cache')

print(f"ULA records: {len(ula_df_total):,}")
print("[PROGRESS] Data Fetch Complete")

Temp tables created
ULA ready: 81,635 records
Recovery ready: 22,041 records
STE weekly metrics ready: 22,140 records
ULA records: 81,635
[PROGRESS] Data Fetch Complete


In [6]:
# =============================================================================
# CELL 6: PERIOD ASSIGNMENT + DATE FILTERING + FLAG CREATION + STE METRICS
# =============================================================================

# --- Filter and relabel ---
ula_df_total = ula_df_total[ula_df_total.lob != 'Core']
ula_df_total['lob'] = 'STE'

ula_df_total['app_date'] = pd.to_datetime(ula_df_total['app_date'])
ula_df_total['book_date'] = pd.to_datetime(ula_df_total['book_date'])
new_recovery['app_date'] = pd.to_datetime(new_recovery['app_date'])
new_recovery['book_date'] = pd.to_datetime(new_recovery['book_date'])

for df in [ula_df_total, new_recovery]:
    df['quarter'] = pd.to_datetime(df[date_col]).dt.to_period('Q')
    df['month'] = pd.to_datetime(df[date_col]).dt.to_period('M')
    df['week'] = pd.to_datetime(df[date_col]).dt.to_period('W-SAT')

    period_key = {'q': 'quarter', 'm': 'month', 'w': 'week'}
    df['period'] = df[period_key[granularity]]

for df in [ula_df_total, new_recovery]:
    mask = (df['period'] >= start_period) & (df['period'] <= end_period)
    df.drop(df[~mask].index, inplace=True)

ula_df_total['book_week'] = ula_df_total['book_week'].astype(str)

ula_df_total[f'{date_col}_str'] = ula_df_total[date_col].astype(str)

# --- STE-specific caps ---
ula_df_total['bbltv'] = ula_df_total.amt_financed / ula_df_total.bbvalue.replace(0, np.nan)
ula_df_total = ula_df_total[
    (ula_df_total.bbltv <= 10.0) |
    (ula_df_total.bbvalue.isna()) |
    (ula_df_total.bbvalue == 0)
]
ula_df_total = ula_df_total[ula_df_total.pti <= 0.6]
ula_df_total = ula_df_total[ula_df_total.total_income <= 200000]
ula_df_total = ula_df_total[ula_df_total['model_score'].notna()]

# Apply model score v4.0/4.1 adjustment to contract-stage model score (cd_model_score)
_ms_adj_mask = (
    ula_df_total['model_score_version'].isin([4.0, 4.1])
    & (ula_df_total['app_date'] < '2026-05-20')
)
ula_df_total.loc[_ms_adj_mask, 'cd_model_score'] = (
    (0.8177 * ula_df_total.loc[_ms_adj_mask, 'cd_model_score']) + 24.5
).astype(int)

print(f"Periods in data: {ula_df_total['period'].nunique()}")
print(f"Period range: {ula_df_total['period'].min()} to {ula_df_total['period'].max()}")
print(f"ULA after caps: {len(ula_df_total):,}")

# --- Flag creation ---
date_col_str = f'{date_col}_str'

ula_df_total['mtn_3_1_flag'] = ula_df_total.mtn_model == 'MTN3.1'
ula_df_total['vehicle_age'] = np.maximum(ula_df_total[date_col_str].str[:4].astype(int) - ula_df_total.model_year, 1/365)
ula_df_total.tradein_value = ula_df_total.tradein_value.fillna(0)
ula_df_total.make = ula_df_total.make.str.upper().str[:3]
ula_df_total.lob_or_bucket = np.select(
    [ula_df_total.lob_or_bucket.isna() & ula_df_total.lob.isin(['Core', 'FRN']),
     ula_df_total.lob_or_bucket.isna() & ~ula_df_total.lob.isin(['Core', 'FRN'])],
    ['D', 'C'], default=ula_df_total.lob_or_bucket)
ula_df_total['continuous_vehicle_age'] = (ula_df_total[date_col_str].str[:4].astype(int)
    + ula_df_total[date_col_str].str[5:7].astype(int) / 12
    - (ula_df_total.model_year - 0.25) - 1)

ula_df_total['pricing_scalar'] = PRICING_SCALAR

warnings.filterwarnings("ignore", category=UserWarning)
ula_df_total.job_company = ula_df_total.job_company.fillna('')
ula_df_total['driver_flag'] = np.where(
    ula_df_total.job_company.str.contains('(LYFT)|(UBER)|(GRUB ?HUB)|(DOOR ?DASH)|(GO ?PUFF)|(POST ?MATE)|(INSTA ?CART)|(DOMINO)|(PAPA J)|(PIZZA)|(JIMMY ?JOHN)|(SELF)'),
    1, 0)
warnings.filterwarnings("default", category=UserWarning)

ula_df_total = ula_df_total.dropna(subset=['lob'])
ula_df_total.cash_down = ula_df_total.cash_down.fillna(0)
ula_df_total.specialty_dealer = ula_df_total.specialty_dealer.fillna('not specialty')
ula_df_total.prev_co_count = ula_df_total.prev_co_count.fillna(0)

# --- NonKMX Flags ---
ula_df_total['ent_fld_flag'] = False
ula_df_total['small_amt_financed_flag'] = (ula_df_total.bbvalue > 0) & (ula_df_total.bbvalue < 5000) & (ula_df_total.amt_financed < 4500)
ula_df_total['zero_cash_down_flag'] = (ula_df_total.cash_down <= 250) & (ula_df_total.tradein_value < 3000)
ula_df_total['high_mileage_vehicle_flag'] = (ula_df_total.mileage >= 100000) & (ula_df_total.cd_model_score < 130)
ula_df_total['high_pti_flag'] = (ula_df_total.pti > 0.2) & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['car_make_penalty_flag'] = ula_df_total.make.isin({'CAD', 'CHR', 'BMW', 'BUI', 'SUB'}) & (ula_df_total.cd_model_score < 130) & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['car_make_benefit_flag'] = (ula_df_total.cd_model_score >= 133) & ula_df_total.make.isin({'HON', 'TOY', 'LEX'}) & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['theft_risk_flag'] = ula_df_total.make.isin({'KIA', 'HYU'}) & (ula_df_total.specialty_dealer != 'Ally') & (ula_df_total.model_year >= 2015) & (ula_df_total.model_year <= 2021) & (ula_df_total[date_col_str] >= '2022-07-01') & (ula_df_total[date_col_str] < '2025-01-01')
ula_df_total['mcy_low_mileage_flag'] = False
ula_df_total['weekend_flag'] = ula_df_total.day_of_week.isin([0, 6])
ula_df_total['weekday_flag'] = ula_df_total.day_of_week.isin(range(1, 6))
ula_df_total['secured_credit_flag'] = ula_df_total.secured_credit_card
ula_df_total['chime_flag'] = ula_df_total.chime_indicator
ula_df_total['nonkmx_chime_flag'] = ula_df_total.nonkmx_chime_indicator
ula_df_total['seasonal_employment_flag'] = ula_df_total['seasonal_employment_flag'] == 1 if 'seasonal_employment_flag' in ula_df_total.columns else (ula_df_total.get('employment', '') == 'seasonal')
ula_df_total['waiter_employment_flag'] = ula_df_total['waiter_employment_flag'] == 1 if 'waiter_employment_flag' in ula_df_total.columns else (ula_df_total.get('employment', '') == 'waiter')
ula_df_total['nonkmx_auth_tradelines_flag'] = ula_df_total.pct_auth_tradelines > 0.2
ula_df_total['prev_co_flag'] = ula_df_total.prev_co_count > 0
ula_df_total['null_fico_w_vantage_flag'] = ((ula_df_total.fico_score < 300) | (ula_df_total.fico_score > 850)) & (ula_df_total.vantage_score >= 300) & (ula_df_total.vantage_score <= 850)
ula_df_total['pricing_change_flag'] = ula_df_total[date_col_str] >= '2024-10-01'
ula_df_total['ent_flag'] = False
ula_df_total['student_loans_cutoff_date'] = ula_df_total[date_col_str] >= '2023-05-01'
ula_df_total['low_pti_flag'] = (ula_df_total.pti <= 0.05) & (ula_df_total.cd_model_score >= 130) & (ula_df_total.cb_flag)
ula_df_total['student_loan_flag'] = 0

# --- KMX Flags ---
ula_df_total['high_sales_price_flag'] = ula_df_total.sale_price > 30000
ula_df_total['kmx_npc_flag'] = False
ula_df_total['high_pti_npc'] = ula_df_total.pti > 0.2
ula_df_total['job_time_flag'] = ula_df_total.employed_months < 6
ula_df_total['low_fico_flag'] = (ula_df_total.fico_score > 300) & (ula_df_total.fico_score < 450)
ula_df_total['high_model_score_flag'] = ula_df_total.cd_model_score >= 146
ula_df_total['low_vantage_flag'] = (ula_df_total.fico_score < 300) & (ula_df_total.vantage_score > 4) & (ula_df_total.vantage_score < 450)
ula_df_total['louisiana_flag'] = ula_df_total.state == 'LA'
ula_df_total['txca_flag'] = ula_df_total.state.isin(['TX', 'CA'])
ula_df_total['georgia_flag'] = ula_df_total.state == 'GA'
ula_df_total['normal_pti_flag'] = ula_df_total.pti <= 0.2
ula_df_total['high_pti_tier_1_flag'] = (ula_df_total.pti > 0.2) & (ula_df_total.pti <= 0.25)
ula_df_total['high_pti_tier_2_flag'] = (ula_df_total.pti > 0.25) & (ula_df_total.pti <= 0.35)
ula_df_total['high_pti_tier_3_flag'] = ula_df_total.pti > 0.35
ula_df_total['existing_dq_flag'] = ula_df_total.existing_dq_count > 0
ula_df_total['kmx_auth_tradelines_flag'] = ula_df_total.pct_auth_tradelines >= 0.14
ula_df_total['low_bureau_flag'] = ((ula_df_total.fico_score > 300) & (ula_df_total.fico_score < 475)) | ((ula_df_total.vantage_score > 4) & (ula_df_total.vantage_score < 450))
ula_df_total['soft_pull_flag'] = ula_df_total.pull_type.isin(['softpull', 'prequal'])
ula_df_total['cd_perc_flag'] = (ula_df_total.sale_price <= 18000) & (ula_df_total.cash_down / ula_df_total.sale_price <= 0.1)
ula_df_total['open_tl_flag'] = ula_df_total.open_tl == 0
ula_df_total['narrowed_soft_pull_flag'] = (ula_df_total.sale_price <= 18000) & (ula_df_total.cash_down / ula_df_total.sale_price <= 0.1) & ula_df_total.soft_pull_flag & (~ula_df_total.job_time_flag)

# --- New/Used determination (age + mileage based) ---
ula_df_total['is_new'] = (
    ula_df_total['model_year'].notna()
    & (pd.to_datetime(ula_df_total['app_date']).dt.year - ula_df_total['model_year'] <= 2)
    & (
        ((ula_df_total['purchase_type'].str.strip().str.lower() == 'used')
         & ula_df_total['mileage'].notna() & (ula_df_total['mileage'] < 500))
        |
        ((ula_df_total['purchase_type'].str.strip().str.lower() == 'new')
         & ((ula_df_total['mileage'].isna()) | (ula_df_total['mileage'] <= 7500)))
    )
).astype(int)

# --- Deduplicate driver flags ---
combined_driver_flag_df = ula_df_total.groupby('account_number').driver_flag.max().reset_index()
ula_df_total = ula_df_total.drop(columns='driver_flag').merge(combined_driver_flag_df, on='account_number')
ula_df_total = ula_df_total.drop(columns='job_company').drop_duplicates()

_pre = len(ula_df_total)
ula_df_total = ula_df_total.drop_duplicates(subset='account_number', keep='first')
if len(ula_df_total) < _pre:
    print(f"  WARNING: Dropped {_pre - len(ula_df_total)} duplicate account_number rows")

# --- Vintage strings ---
ula_df_total['vintage'] = format_vintage(ula_df_total['period'])
new_recovery['vintage'] = format_vintage(new_recovery['period'])

print(f"ULA after refinement: {len(ula_df_total):,}")

# --- Build ste_filtered from weekly data ---
ste_filtered = ste_weekly_raw.copy()
ste_date_col = 'application_received_dtm' if granularity == 'w' else 'book_date'
ste_filtered[ste_date_col] = pd.to_datetime(ste_filtered[ste_date_col])
ste_filtered['period'] = ste_filtered[ste_date_col].dt.to_period(period_freq)
ste_filtered = ste_filtered[(ste_filtered.period >= start_period) & (ste_filtered.period <= end_period)]
ste_filtered['vintage'] = format_vintage(ste_filtered['period'])

ste_filtered = ste_filtered[ste_filtered['con_pti_back'] <= 0.6]
ste_filtered = ste_filtered[ste_filtered['total_income'] <= 200000]
ste_filtered['bbltv'] = ste_filtered['con_amount_financed_back'] / ste_filtered['bb_value'].replace(0, np.nan)
ste_filtered = ste_filtered[
    (ste_filtered['bbltv'] <= 10.0) |
    (ste_filtered['bb_value'].isna()) |
    (ste_filtered['bb_value'] == 0)
]

print(f"STE weekly after caps: {len(ste_filtered):,}")

# --- Merge ULA model_score and model_score_version onto weekly data ---
ula_merge_cols = ula_df_total[['account_number', 'model_score', 'model_score_version']].drop_duplicates(subset='account_number', keep='first')
_pre_wk = len(ste_filtered)
ste_filtered = ste_filtered.merge(ula_merge_cols, on='account_number', how='left')
if len(ste_filtered) != _pre_wk:
    print(f"  WARNING: Weekly merge changed row count from {_pre_wk:,} to {len(ste_filtered):,}, deduplicating")
    ste_filtered = ste_filtered.drop_duplicates(subset='account_number', keep='first')

# Apply model score v4.0/4.1 adjustment to weekly contract model score
_wk_adj_mask = (
    ste_filtered['model_score_version'].isin([4.0, 4.1])
    & (pd.to_datetime(ste_filtered['application_received_dtm']) < '2026-05-20')
)
ste_filtered.loc[_wk_adj_mask, 'con_risk_model_score'] = (
    (0.8177 * ste_filtered.loc[_wk_adj_mask, 'con_risk_model_score']) + 24.5
).astype(int)

print(f"Weekly rows with model_score: {ste_filtered['model_score'].notna().sum():,} / {len(ste_filtered):,}")

# --- New/Used determination for weekly data ---
ste_filtered['is_new'] = (
    ste_filtered['purchase_year'].notna()
    & (pd.to_datetime(ste_filtered['application_received_dtm']).dt.year - pd.to_numeric(ste_filtered['purchase_year'], errors='coerce') <= 2)
    & (
        ((ste_filtered['purchase_type'].str.strip().str.lower() == 'used')
         & ste_filtered['purchase_odometer'].notna() & (pd.to_numeric(ste_filtered['purchase_odometer'], errors='coerce') < 500))
        |
        ((ste_filtered['purchase_type'].str.strip().str.lower() == 'new')
         & ((ste_filtered['purchase_odometer'].isna()) | (pd.to_numeric(ste_filtered['purchase_odometer'], errors='coerce') <= 7500)))
    )
).astype(int)

# --- Pool population counts ---
for pool_name, pool_cfg in POOLS.items():
    ula_n = len(pool_cfg['ula_filter'](ula_df_total)) if pool_cfg['ula_filter'] else len(ula_df_total)
    wk_n = len(pool_cfg['weekly_filter'](ste_filtered)) if pool_cfg['weekly_filter'] else len(ste_filtered)
    print(f"  {pool_name}: ULA={ula_n:,}, Weekly={wk_n:,}")

print("[PROGRESS] Data Prep Complete")

Periods in data: 11
Period range: 2025-10 to 2026-08
ULA after caps: 79,251
ULA after refinement: 20,741
STE weekly after caps: 20,737
Weekly rows with model_score: 20,640 / 20,737
  Vanilla: ULA=20,741, Weekly=20,737
  New Cars: ULA=2,608, Weekly=2,608
  Downcredit Used: ULA=15,393, Weekly=15,336
  Upcredit Used: ULA=2,740, Weekly=2,715
  Downcredit: ULA=17,479, Weekly=17,412
  Upcredit: ULA=3,262, Weekly=3,228
[PROGRESS] Data Prep Complete


In [7]:
# =============================================================================
# CELL 7: POOL-LEVEL RAGU SCORING
# =============================================================================

mean_unit_loss = MODEL_PARAMS['mean_unit_loss']
unit_loss_to_model_score = MODEL_PARAMS['unit_loss_to_model_score']


def _build_ste_metrics(g):
    w = g['con_amount_financed_back']
    ms_valid = g['con_risk_model_score'].notnull()
    ltv_valid = g['bbltv'].notnull()
    return pd.Series({
        'model_score_wtd': (g.loc[ms_valid, 'con_risk_model_score'] * w[ms_valid]).sum() / w[ms_valid].sum() if ms_valid.any() else np.nan,
        'ltv_wtd': (g.loc[ltv_valid, 'bbltv'] * w[ltv_valid]).sum() / w[ltv_valid].sum() if ltv_valid.any() else np.nan,
        'apr_wtd': (g['con_apr'] * w).sum() / w.sum(),
        'amt_financed_total': w.sum(),
        'n_accounts': len(g),
    })


def get_ragu_score(vintage, lob, ula_df_sub, new_recovery, ms_df, baseline_config,
                   leave_out='None', ste_metrics_df=None):
    baseline_ltv = baseline_config['ltv']
    new_baseline_recovery_unadjusted_pct = baseline_config['new_recovery_unadjusted']
    baseline_apr = baseline_config['apr']

    ula_df = ula_df_sub[(ula_df_sub.vintage == vintage) & (ula_df_sub.lob == lob)].copy()

    if len(ula_df) == 0:
        return None

    ula_df = get_ula_multiplier_nonkmx(ula_df, leave_out)

    ula_df = ula_df[['account_number', date_col, 'bbvalue', 'sale_price', 'amt_financed',
                     'lob_or_bucket', 'lob', 'loss_multiplier', 'apr']]

    nr = new_recovery[['account_number', 'new_recovery_multiplier']].drop_duplicates(
        subset='account_number', keep='first')
    mix_df = ula_df.merge(nr.rename(columns={'new_recovery_multiplier': 'recovery_multiplier'}),
                          on='account_number', how='left').drop_duplicates(
                          subset='account_number', keep='first')

    mix_df['ltv'] = mix_df.amt_financed / mix_df.bbvalue

    bb_populated_df = mix_df[mix_df['bbvalue'].notna() & (mix_df['bbvalue'] > 0)]
    if len(bb_populated_df) == 0:
        return None

    full_pop_metrics = bb_populated_df.groupby('lob').apply(
        weighted_average_and_sum,
        ['loss_multiplier', 'ltv', 'bbvalue'],
        include_groups=False
    )

    apr_metrics = mix_df.groupby('lob').apply(
        weighted_average_and_sum, 'apr', include_groups=False
    ).drop(columns='amt_financed')

    recovery_df = mix_df[mix_df['bbvalue'].notna() & (mix_df['bbvalue'] > 0) & mix_df['recovery_multiplier'].notna()]
    recovery_df = recovery_df.copy()
    recovery_df['recovery_unadjusted_multiplier'] = recovery_df['recovery_multiplier']
    recovery_metrics = recovery_df.groupby('lob').apply(
        weighted_average_and_sum,
        ['recovery_unadjusted_multiplier'],
        include_groups=False
    )
    recovery_metrics = recovery_metrics.drop(columns='amt_financed')

    grouped_mix_df = full_pop_metrics.join(recovery_metrics).join(apr_metrics)

    vintage_ms_df = ms_df.loc[ms_df['period'] == vintage, ['lob', 'model_score']].copy()

    index_name = grouped_mix_df.index.name
    if isinstance(index_name, str) and index_name in grouped_mix_df.columns:
        grouped_mix_df = grouped_mix_df.reset_index(drop=True)
    else:
        grouped_mix_df = grouped_mix_df.reset_index()

    full_df = grouped_mix_df.merge(vintage_ms_df, on='lob')

    if len(full_df) == 0:
        return None

    full_df['est_unit_loss'] = mean_unit_loss
    full_df['unit_loss_score'] = full_df.model_score + (1 - full_df.loss_multiplier) * full_df.est_unit_loss / unit_loss_to_model_score
    full_df = full_df.set_index('lob')

    full_df['ms_original'] = full_df.model_score.copy()
    full_df['baselined_unadjusted_recovery'] = (full_df.recovery_unadjusted_multiplier / new_baseline_recovery_unadjusted_pct).copy()

    full_df['gross_loss_impact'] = full_df['unit_loss_score'] - full_df['ms_original']
    full_df['recovery_impact'] = (full_df['unit_loss_score'] * full_df['est_unit_loss']
        * full_df['recovery_unadjusted_multiplier'] * (full_df['baselined_unadjusted_recovery'] - 1))
    full_df['ltv_impact'] = ((baseline_ltv / full_df['ltv']) - 1) * 17
    full_df['apr_impact'] = (baseline_apr - full_df['apr']) / 0.01 * 0.7
    full_df['ragu_score'] = (
        (1 - full_df.est_unit_loss * full_df.recovery_unadjusted_multiplier) * full_df.unit_loss_score
        + full_df.est_unit_loss * full_df.recovery_unadjusted_multiplier
        * full_df.unit_loss_score * full_df.baselined_unadjusted_recovery
        + full_df['ltv_impact']
        + full_df['apr_impact'])

    if ste_metrics_df is not None:
        ste_row = ste_metrics_df[ste_metrics_df.vintage == vintage]
        if len(ste_row) > 0:
            ste_row = ste_row.iloc[0]
            full_df['ms_original'] = ste_row['model_score_wtd']
            full_df['ltv'] = ste_row['ltv_wtd']
            full_df['apr'] = ste_row['apr_wtd']
            full_df['ltv_impact'] = ((baseline_ltv / full_df['ltv']) - 1) * 17
            full_df['apr_impact'] = (baseline_apr - full_df['apr']) / 0.01 * 0.7
            full_df['ragu_score'] = (
                (1 - full_df.est_unit_loss * full_df.recovery_unadjusted_multiplier) * full_df.unit_loss_score
                + full_df.est_unit_loss * full_df.recovery_unadjusted_multiplier
                * full_df.unit_loss_score * full_df.baselined_unadjusted_recovery
                + full_df['ltv_impact']
                + full_df['apr_impact'])

    full_df['vintage'] = vintage
    return full_df


# --- Main loop: iterate over pools ---
all_vintages = sorted(ula_df_total['vintage'].unique())
all_pool_results = []

for pool_name, pool_cfg in POOLS.items():
    ula_subset = pool_cfg['ula_filter'](ula_df_total) if pool_cfg['ula_filter'] else ula_df_total.copy()
    wk_subset = pool_cfg['weekly_filter'](ste_filtered) if pool_cfg['weekly_filter'] else ste_filtered.copy()

    if len(ula_subset) == 0:
        print(f'{pool_name}: no ULA loans found, skipping')
        continue

    ms_subset = ula_subset.groupby(['period', 'lob']).apply(
        weighted_average_and_sum, 'model_score', include_groups=False
    ).reset_index()
    ms_subset['period'] = format_vintage(ms_subset['period'])

    pool_ste_metrics = wk_subset.groupby('vintage').apply(_build_ste_metrics).reset_index()

    lob_results = []
    baseline_config = BASELINES['STE']
    excluded = EXCLUDED_VINTAGES.get('STE', set())

    for vintage in all_vintages:
        if vintage in excluded:
            continue
        try:
            result = get_ragu_score(
                vintage, 'STE', ula_subset, new_recovery, ms_subset, baseline_config,
                ste_metrics_df=pool_ste_metrics
            )
            if result is not None:
                lob_results.append(result)
        except Exception as e:
            print(f'  Error: {pool_name} / {vintage}: {e}')

    if not lob_results:
        print(f'{pool_name}: no scoreable vintages')
        continue

    pool_df = pd.concat(lob_results, ignore_index=False).reset_index()

    rollup_metrics = [
        'ms_original', 'gross_loss_impact', 'recovery_impact',
        'ltv_impact', 'apr_impact', 'ragu_score', 'ltv', 'apr',
    ]
    rollup = pool_df.groupby('vintage').apply(
        weighted_average_and_sum, rollup_metrics, include_groups=False
    ).reset_index()
    rollup['lob'] = pool_name

    all_pool_results.append(rollup)
    n_vintages = rollup.vintage.nunique()
    print(f'{pool_name}: {n_vintages} vintages scored ({len(ula_subset):,} ULA loans, {len(wk_subset):,} weekly records)')

all_df = pd.concat(all_pool_results, ignore_index=True)
print(f'\nTotal results: {len(all_df)} rows across {all_df.lob.nunique()} pools and {all_df.vintage.nunique()} vintages')
print('[PROGRESS] Pool RAGU Scoring Complete')

Vanilla: 11 vintages scored (20,741 ULA loans, 20,737 weekly records)
New Cars: 11 vintages scored (2,608 ULA loans, 2,608 weekly records)
Downcredit Used: 11 vintages scored (15,393 ULA loans, 15,336 weekly records)
Upcredit Used: 11 vintages scored (2,740 ULA loans, 2,715 weekly records)
Downcredit: 11 vintages scored (17,479 ULA loans, 17,412 weekly records)
Upcredit: 11 vintages scored (3,262 ULA loans, 3,228 weekly records)

Total results: 66 rows across 6 pools and 11 vintages
[PROGRESS] Pool RAGU Scoring Complete


In [8]:
# =============================================================================
# CELL 8: DISPLAY + EXCEL EXPORT
# =============================================================================

METRIC_ROWS = [
    ('Model Score',       'ms_original'),
    ('Gross Loss Impact', 'gross_loss_impact'),
    ('Recovery Impact',   'recovery_impact'),
    ('LTV Impact',        'ltv_impact'),
    ('APR Impact',        'apr_impact'),
    ('RAGU Score',        'ragu_score'),
    ('Amount Financed',   'amt_financed'),
    ('Weighted LTV',      'ltv'),
    ('Weighted APR',      'apr'),
]

EXCEL_OUTPUT = '../output/ste_ragu_vanillas.xlsx'
EXCEL_SHEET_MAP = {'q': 'Data Tables (Q)', 'm': 'Data Tables (M)', 'w': 'Data Tables (W)'}
sheet_name = EXCEL_SHEET_MAP[granularity]

sorted_vintages = sorted(all_df['vintage'].unique())
all_export_pools = list(POOLS.keys())

# --- Display summary per pool ---
pd.set_option('display.float_format', '{:.4f}'.format)
for pool_name in all_export_pools:
    pool_data = all_df[all_df.lob == pool_name]
    if len(pool_data) == 0:
        continue
    print(f'\n=== {pool_name} ===')
    pivot = pool_data.set_index('vintage')[['ms_original', 'gross_loss_impact',
        'recovery_impact', 'ltv_impact', 'apr_impact', 'ragu_score', 'amt_financed']].T
    display(pivot)

# --- Excel export ---
if os.path.exists(EXCEL_OUTPUT):
    wb = openpyxl.load_workbook(EXCEL_OUTPUT)
    if sheet_name in wb.sheetnames:
        del wb[sheet_name]
    ws = wb.create_sheet(sheet_name)
else:
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = sheet_name

current_row = 1
for pool_name in all_export_pools:
    pool_data = all_df[all_df.lob == pool_name].set_index('vintage')
    if len(pool_data) == 0:
        continue

    ws.cell(row=current_row, column=1, value=pool_name)
    for col_idx, v in enumerate(sorted_vintages, start=2):
        ws.cell(row=current_row, column=col_idx, value=v)
    current_row += 1

    for label, col_key in METRIC_ROWS:
        ws.cell(row=current_row, column=1, value=label)
        for col_idx, v in enumerate(sorted_vintages, start=2):
            if v in pool_data.index:
                ws.cell(row=current_row, column=col_idx, value=pool_data.loc[v, col_key])
        current_row += 1
    current_row += 1

wb.save(EXCEL_OUTPUT)
print(f'\nSaved to {EXCEL_OUTPUT} (sheet: {sheet_name})')
print(f'  {len(all_export_pools)} pools x {len(sorted_vintages)} periods')
print('[PROGRESS] Excel Export Complete')


=== Vanilla ===


vintage,2025 M10,2025 M11,2025 M12,2026 M01,2026 M02,2026 M03,2026 M04,2026 M05,2026 M06,2026 M07,2026 M08
ms_original,131.0213,131.3468,131.6024,132.1922,133.0075,134.0049,134.5653,135.6894,136.4966,137.3371,137.6354
gross_loss_impact,-0.3933,-0.4233,-0.4436,-0.3777,-0.3999,-0.3620,-0.2781,-0.4017,-0.5681,-0.5086,-0.0608
recovery_impact,2.9817,2.1450,1.9836,2.0672,1.2730,1.1276,2.2900,3.4352,4.0146,4.3950,3.4257
ltv_impact,4.6568,4.3505,3.9599,4.4326,4.9778,5.5671,5.8010,6.4230,6.1696,6.3675,6.1588
apr_impact,0.9450,1.1155,1.0915,1.0879,1.1751,1.5047,1.6723,1.7831,1.8827,2.0487,1.7951
ragu_score,139.2117,138.5346,138.1938,139.4022,140.0335,141.8454,144.0512,146.9294,147.9956,149.6410,148.9542
amt_financed,31718897.3600,46957669.3800,48436705.3600,41847911.8400,51674677.0100,107724085.9800,90227374.9500,72604977.9000,82073268.2500,58437607.4000,21018873.9800



=== New Cars ===


vintage,2025 M10,2025 M11,2025 M12,2026 M01,2026 M02,2026 M03,2026 M04,2026 M05,2026 M06,2026 M07,2026 M08
ms_original,131.5282,131.0423,132.1682,132.2197,133.0350,133.5622,135.1753,135.6475,136.5483,138.2147,138.5005
gross_loss_impact,-0.3513,-0.3973,-0.6361,-0.7414,-0.3369,-0.3114,-0.3485,-0.5990,-0.7173,-0.4330,0.0681
recovery_impact,13.0068,11.0435,11.1667,10.3995,11.6718,10.2525,10.8999,11.0146,11.6605,11.3481,11.4916
ltv_impact,4.3124,4.4313,4.4059,3.7159,4.7940,5.1596,5.6489,6.3076,6.2229,6.1874,6.0556
apr_impact,2.5848,2.3254,2.8328,2.4465,2.6086,2.6020,3.2292,3.3276,3.3298,3.1417,2.7072
ragu_score,151.0808,148.4452,149.9375,148.0401,151.7724,151.2649,154.6048,155.6983,157.0443,158.4589,158.8229
amt_financed,6615151.5500,8277091.6400,7352325.4700,6273960.8800,5858883.7700,11892408.7600,14244101.6200,19209750.7300,24620167.4000,18390121.6500,5052119.6100



=== Downcredit Used ===


...


=== Downcredit ===


vintage,2025 M10,2025 M11,2025 M12,2026 M01,2026 M02,2026 M03,2026 M04,2026 M05,2026 M06,2026 M07,2026 M08
ms_original,130.2811,130.6184,130.9259,131.4687,131.8008,132.6294,132.9701,133.5465,133.9703,134.0945,134.1602
gross_loss_impact,-0.4711,-0.4150,-0.4501,-0.4350,-0.4698,-0.4445,-0.3642,-0.5566,-0.6162,-0.6377,-0.2397
recovery_impact,2.9812,2.1387,2.0311,1.9885,1.2675,1.0523,2.2292,3.2115,3.9486,3.8549,3.3130
ltv_impact,4.5258,4.1607,3.7552,4.2221,4.7317,5.3623,5.5305,5.9560,5.7058,5.7699,5.3238
apr_impact,0.8191,1.0031,0.9843,0.9469,0.8738,1.0240,1.1679,1.1666,1.1207,0.7914,0.2782
ragu_score,138.1362,137.5059,137.2464,138.1912,138.2040,139.6247,141.5362,143.3245,144.1291,143.8729,142.9086
amt_financed,30007458.2800,44266632.0800,45548957.5800,39158647.1400,46307784.7300,93024872.2100,75801917.1200,56879580.1300,59712949.5500,38581810.4100,13738704.2600



=== Upcredit ===


vintage,2025 M10,2025 M11,2025 M12,2026 M01,2026 M02,2026 M03,2026 M04,2026 M05,2026 M06,2026 M07,2026 M08
ms_original,142.7971,142.8328,142.1830,142.7859,143.4686,142.7037,142.9101,143.4436,143.2588,143.6069,144.1548
gross_loss_impact,0.9708,-0.5594,-0.3413,0.4568,0.2029,0.1602,0.1739,0.1586,-0.4394,-0.2577,0.2768
recovery_impact,2.8978,2.2349,1.0681,3.3095,1.3156,1.6365,2.6281,4.3126,4.1893,5.5810,3.6394
ltv_impact,7.2450,8.0464,7.8229,8.0501,7.3582,6.9575,7.3347,8.2833,7.5065,7.6245,7.7102
apr_impact,2.9485,2.8886,2.7682,3.1535,3.7875,4.5447,4.3110,4.0139,3.9225,4.4824,4.5819
ragu_score,156.8592,155.4433,153.5009,157.7558,156.1329,156.0053,157.3578,160.2121,158.4376,161.0370,160.1543
amt_financed,1711439.0800,2691037.3000,2887747.7800,2689264.7000,5366892.2800,14699213.7700,14425457.8300,15725397.7700,22360318.7000,19855796.9900,7280169.7200



Saved to ../output/ste_ragu_vanillas.xlsx (sheet: Data Tables (M))
  6 pools x 11 periods
[PROGRESS] Excel Export Complete
